# RANSAC multi-view hand keypoint triangulation (no WiLoR/MANO)

Pure-geometric companion to `wilor_hand_reconstruction_v2.ipynb` (left untouched).
Instead of fitting a MANO mesh per camera, this notebook:

1. Runs a robust per-camera detection pipeline (YOLO + MediaPipe detection union,
   temporal tracking, majority-vote handedness resolution, two-signal occlusion
   flagging) to get, for each of the 7 cameras and each frame, MediaPipe's 21 2D
   hand landmarks per hand (when available), plus presence/occlusion flags.
2. Triangulates each of the 21 landmarks into 3D using the calibrated camera
   projection matrices, with a RANSAC step (exhaustive camera pairs, reprojection-
   error inlier counting on the wrist joint) that automatically drops views where a
   hand is occluded or mislabeled.
3. Visualizes the resulting 3D hand skeletons in viser.

Generated by `v3/notebooks/build_hand_notebook.py`, which is the source of truth --
edit that script and re-run `python -m v3.notebooks.build_hand_notebook`, don't edit
this notebook directly. Shared calibration/triangulation/smoothing/visualization
logic lives in the `v3` package (also used by `body_keypoint_reconstruction.ipynb`).

Runs entirely in `oak_env`.

In [ ]:
import os
import time

import numpy as np
import torch
from ultralytics import YOLO

from v3.calibration import load_calibration
from v3.keypoints_io import load_extracted_keypoints
from v3.hand_detection import run_hand_detection
from v3.triangulation import undistort_keypoints, ransac_triangulate_sequence
from v3.smoothing import smooth_sequence, jitter_flags, savgol_and_upsample_2x
from v3.skeletons import HAND_CONNECTIONS
from v3 import visualization as viz

WILOR_DIR = '/home/fmalmb/CODE/WiLoR'
H5_PATH = '/data/fmalmb/DATA/tmp/Testdata.h5'
IMG_DIR = '/tmp/wilor_demo_input'  # produced by /tmp/extract_remaining_cams.py etc.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

detector = YOLO(os.path.join(WILOR_DIR, 'pretrained_models/detector.pt'))
detector = detector.to(DEVICE)
print(f'YOLO detector loaded on {DEVICE}')

In [ ]:
CAMERA_CALIB, CAMERA_WORLD_TRANSFORMS, PROJECTION_MATRICES, camera_ids = load_calibration(H5_PATH)
N_CAMS = len(camera_ids)

In [ ]:
keypoints_2d = load_extracted_keypoints('extracted_2d_keypoints.json')

In [ ]:
# --- Stage 1: per-camera detection union, tracking, occlusion flagging --------------
# YOLO + MediaPipe detection union, temporal tracking, majority-vote handedness
# resolution, two-signal occlusion flagging (v3.hand_detection.run_hand_detection).
# Each detection that has a MediaPipe match also carries that hand's 21 landmarks,
# which is the actual 2D input to triangulation below.

FRAME_OFFSET = 40
N_FRAMES = 50
N_JOINTS = 21

# hand_idx: 0 = left, 1 = right (global hand identity -- see notes_on_wilor_implementation.md)
detection = run_hand_detection(detector, keypoints_2d, camera_ids, IMG_DIR,
                                frame_offset=FRAME_OFFSET, n_frames=N_FRAMES)
keypoints_px = detection['keypoints_px']
keypoints_conf = detection['keypoints_conf']
presence = detection['presence']
occlusion = detection['occlusion']

np.savez_compressed(
    'ransac_keypoint_detections.npz',
    keypoints_px=keypoints_px, keypoints_conf=keypoints_conf,
    presence=presence, occlusion=occlusion,
    camera_ids=np.array(camera_ids), frame_indices=np.arange(FRAME_OFFSET, FRAME_OFFSET + N_FRAMES),
)
print('Saved ransac_keypoint_detections.npz')

In [ ]:
# --- Stage 2: RANSAC multi-view triangulation ----------------------------------------
RANSAC_REPROJ_THRESH_PX = 50.0  # wrist reprojection-error inlier threshold; tune from the
                                 # residual stats printed below (calibration baseline is
                                 # several tens of px even for "correct" triangulations).
RANSAC_HYSTERESIS_INLIER_MARGIN = 0   # keep the previous frame's inlier camera set unless a new
                                       # set beats it by more than this many inliers
RANSAC_HYSTERESIS_ERR_FACTOR = 1.5    # ...and its mean reprojection error isn't worse by more
                                       # than this factor

keypoints_px_undist = undistort_keypoints(keypoints_px, CAMERA_CALIB, camera_ids)

result = ransac_triangulate_sequence(
    keypoints_px_undist, keypoints_conf, presence, PROJECTION_MATRICES, camera_ids,
    seed_joint_idx=0, occlusion=occlusion,
    reproj_thresh_px=RANSAC_REPROJ_THRESH_PX,
    hysteresis_inlier_margin=RANSAC_HYSTERESIS_INLIER_MARGIN,
    hysteresis_err_factor=RANSAC_HYSTERESIS_ERR_FACTOR,
)
joints3d = result['joints3d']
valid = result['valid']
n_inliers = result['n_inliers']
reproj_err = result['reproj_err']
wrist_residuals = result['seed_residuals']
hysteresis_overrides = result['hysteresis_overrides']
n_valid_frames = result['n_valid_frames']

for hi, label in enumerate(['left', 'right']):
    n_valid = int(valid[:, hi].sum())
    print(f'{label} hand: {n_valid}/{N_FRAMES} frames valid')
    if n_valid:
        ni = n_inliers[valid[:, hi], hi]
        print(f'  inlier cams: min={ni.min()}, mean={ni.mean():.2f}, max={ni.max()}')
        print(f'  mean reprojection error: {np.nanmean(reproj_err[valid[:, hi], hi, :]):.1f}px')
        print(f'  hysteresis kept previous camera set: {hysteresis_overrides[hi]}/{n_valid_frames[hi]} valid frames')

print(f'wrist RANSAC residual (px): mean={wrist_residuals.mean():.1f}, '
      f'median={np.median(wrist_residuals):.1f}, max={wrist_residuals.max():.1f}')

In [ ]:
# --- Stage 2c: Kalman/RTS smoothing of joint trajectories -----------------------------
# Constant-velocity Kalman filter + Rauch-Tung-Striebel (RTS) backward smoother, applied
# independently to each joint's 3D trajectory (v3.smoothing.smooth_sequence). Frames with
# no RANSAC solution (valid=False) get pure predictions, filled in by the backward pass
# using later valid frames. Measurement variance is inflated for n_inliers==2
# triangulations, which have larger depth uncertainty.
KALMAN_PROCESS_STD = 0.03     # m / frame^2 acceleration-noise std (raise if smoothing lags fast motion)
KALMAN_MEAS_STD_BASE = 0.01   # m position-measurement std for an n_inliers>=3 triangulation
KALMAN_MEAS_STD_2CAM = 0.015  # m position-measurement std for an n_inliers==2 triangulation

joints3d_smooth, valid_smooth = smooth_sequence(
    joints3d, valid, n_inliers,
    process_std=KALMAN_PROCESS_STD,
    meas_std_base=KALMAN_MEAS_STD_BASE,
    meas_std_2cam=KALMAN_MEAS_STD_2CAM,
    min_inliers_base=3,
)

for hi, label in enumerate(['left', 'right']):
    vi = valid[:, hi]
    if not vi.any():
        continue
    delta = np.linalg.norm(joints3d[vi, hi, 0] - joints3d_smooth[vi, hi, 0], axis=-1)
    print(f'{label} hand wrist: raw-vs-smoothed correction (cm): '
          f'mean={delta.mean() * 100:.2f}, median={np.median(delta) * 100:.2f}, max={delta.max() * 100:.2f}')

In [ ]:
# --- Stage 2d: jitter flagging (raw RANSAC vs Kalman/RTS-smoothed wrist) ---------------
# Frames where the raw RANSAC wrist position deviates sharply from the Kalman/RTS-smoothed
# trajectory are flagged -- a sign that frame's RANSAC camera-set choice was an outlier,
# now corrected by the smoother (v3.smoothing.jitter_flags).
JITTER_MAD_FACTOR = 4.0  # flag residuals > median + this many MADs above it

jitter_flag, wrist_jitter_residual = jitter_flags(
    joints3d, joints3d_smooth, valid, seed_joint_idx=0, mad_factor=JITTER_MAD_FACTOR)

for hi, label in enumerate(['left', 'right']):
    valid_idx = np.where(valid[:, hi])[0]
    if len(valid_idx) == 0:
        continue
    res = wrist_jitter_residual[valid_idx, hi]
    med = np.median(res)
    mad = np.median(np.abs(res - med)) + 1e-9
    thresh = med + JITTER_MAD_FACTOR * mad
    flagged = valid_idx[res > thresh]

    print(f'{label} hand wrist raw-vs-smoothed residual (cm): median={med * 100:.2f}, '
          f'mad={mad * 100:.2f}, threshold={thresh * 100:.2f}, max={res.max() * 100:.2f}')
    print(f'  flagged frames ({len(flagged)}/{len(valid_idx)}, global idx): '
          f'{(flagged + FRAME_OFFSET).tolist()}')

In [ ]:
# --- Stage 2e: Savitzky-Golay smoothing + 2x frame-rate interpolation -----------------
# Apply a Savitzky-Golay filter (local polynomial fit) to the Kalman/RTS-smoothed
# trajectories for additional noise reduction without the lag a low-pass filter would add,
# then upsample to 2x the original frame rate via cubic-spline interpolation of the now-smooth
# curves -- doubling the temporal resolution of the 3D reconstruction
# (v3.smoothing.savgol_and_upsample_2x).
SG_WINDOW = 7  # frames (must be odd and <= the valid span length)
SG_POLYORDER = 3

sg_result = savgol_and_upsample_2x(joints3d_smooth, valid_smooth, sg_window=SG_WINDOW, sg_polyorder=SG_POLYORDER)
joints3d_sg = sg_result['joints3d_sg']
joints3d_2x = sg_result['joints3d_2x']
valid_2x = sg_result['valid_2x']
N_FRAMES_2X = sg_result['n_frames_2x']
nearest_orig_2x = sg_result['nearest_orig_2x']

# Nearest-original-frame upsampling of per-frame diagnostics, for visualization at 2x rate.
jitter_flag_2x = jitter_flag[nearest_orig_2x]
n_inliers_2x = n_inliers[nearest_orig_2x]

for hi, label in enumerate(['left', 'right']):
    vi = valid_smooth[:, hi]
    if not vi.any():
        continue
    delta = np.linalg.norm(joints3d_smooth[vi, hi, 0] - joints3d_sg[vi, hi, 0], axis=-1)
    n2x = int(valid_2x[:, hi].sum())
    print(f'{label} hand: Savitzky-Golay correction on wrist (cm): mean={delta.mean() * 100:.3f}, '
          f'max={delta.max() * 100:.3f}; {n2x}/{N_FRAMES_2X} frames valid at 2x rate')

In [ ]:
# --- Stage 3: viser visualization -----------------------------------------------------
import viser

HAND_NAMES = {0: 'left', 1: 'right'}
HAND_COLORS = {0: (180, 220, 255), 1: (255, 200, 170)}
LOW_CONFIDENCE_COLOR = (140, 140, 140)
JITTER_COLOR = (230, 40, 40)  # frames where raw RANSAC wrist deviated sharply from the
                               # Kalman/RTS-smoothed trajectory (jitter_flag above)

# Origin: centroid of all valid joints (both hands) in the first valid frame.
ORIGIN_WORLD = viz.compute_origin(joints3d, valid)


def main():
    server = viser.ViserServer()
    running = True
    try:
        server.scene.set_up_direction('+z')
        server.gui.configure_theme(dark_mode=True)
        viz.setup_cameras(server, CAMERA_CALIB, CAMERA_WORLD_TRANSFORMS, ORIGIN_WORLD)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser: open the URL shown above')
        print(f'Origin offset (calibration world): {ORIGIN_WORLD}')

        stop_button = server.gui.add_button('Stop viewer')
        frame_slider = server.gui.add_slider('Frame (2x rate)', min=0, max=N_FRAMES_2X - 1, step=1, initial_value=0)
        play_checkbox = server.gui.add_checkbox('Play (loop)', initial_value=True)
        fps_slider = server.gui.add_slider('Playback FPS', min=1, max=60, step=1, initial_value=20)

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        last_advance = time.time()

        while running:
            fi = frame_slider.value
            for hi in range(2):
                name = f'/hands/{HAND_NAMES[hi]}'
                if valid_2x[fi, hi]:
                    if jitter_flag_2x[fi, hi]:
                        color = JITTER_COLOR
                    elif n_inliers_2x[fi, hi] >= 3:
                        color = HAND_COLORS[hi]
                    else:
                        color = LOW_CONFIDENCE_COLOR
                    viz.add_skeleton(server, name, joints3d_2x[fi, hi], HAND_CONNECTIONS, color,
                                      origin_world=ORIGIN_WORLD)
                else:
                    viz.add_empty_skeleton(server, name)

            if play_checkbox.value:
                now = time.time()
                if now - last_advance >= 1.0 / fps_slider.value:
                    frame_slider.value = (fi + 1) % N_FRAMES_2X
                    last_advance = now

            time.sleep(0.02)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()